In [1]:
import torch
print(torch.cuda.is_available())

False


In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
from moviepy.video.io.VideoFileClip import VideoFileClip
import os
import glob
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import math 
import re 

import imageio_ffmpeg
import os

os.environ["IMAGEIO_FFMPEG_EXE"] = imageio_ffmpeg.get_ffmpeg_exe()


In [3]:

model_path = "HuggingFaceTB/SmolVLM2-256M-Video-Instruct"

processor = AutoProcessor.from_pretrained(model_path)

model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.float32,
    _attn_implementation="eager"
).to("cpu")

model.eval()



SmolVLMForConditionalGeneration(
  (model): SmolVLMModel(
    (vision_model): SmolVLMVisionTransformer(
      (embeddings): SmolVLMVisionEmbeddings(
        (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
        (position_embedding): Embedding(1024, 768)
      )
      (encoder): SmolVLMEncoder(
        (layers): ModuleList(
          (0-11): 12 x SmolVLMEncoderLayer(
            (self_attn): SmolVLMVisionAttention(
              (k_proj): Linear(in_features=768, out_features=768, bias=True)
              (v_proj): Linear(in_features=768, out_features=768, bias=True)
              (q_proj): Linear(in_features=768, out_features=768, bias=True)
              (out_proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): SmolVLMVisionMLP(
              (activation_fn): PytorchGELUTanh()
              (fc1): Linear(in_features=768, out_

In [4]:
# THIS HAS TO CHANGE - HEAVY EXPERIMENTATION
PROMPT_EVENT_ONLY = """ 
Describe the single most important event in this video segment in one concise sentence. Focus on what is happening and who is involved.
"""

In [5]:

# this function runs the model per input chunk 
# need to define number of frames used per chunk + number of output tokens
def run_vlm_on_video(video_path, prompt, num_frames=8, max_new_tokens=128):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "video", "path": video_path},
                {"type": "text", "text": prompt}
            ]
        }
    ]




    inputs = processor.apply_chat_template(
    messages,
    num_frames = num_frames,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    ).to(model.device) # Considering the size of the model, and the number of frames we are sampling are few. This is ultimately a choice that you must make
    
    
    
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens
        )

    generated_only = generated_ids[:, inputs["input_ids"].shape[1]:]

    answer = processor.batch_decode(
        generated_only,
        skip_special_tokens=True
    )[0]

    return answer.strip()


In [6]:
# splitting the entire video into chunks of lenght x seconds


def split_video_into_chunks(video_path, output_dir, chunk_length=8):
    os.makedirs(output_dir, exist_ok=True)

    # If chunks already exist, reuse them
    existing_chunks = sorted(glob.glob(os.path.join(output_dir, "chunk_*.mp4")))
    if existing_chunks:
        chunk_paths = []

        for chunk_path in existing_chunks:
            filename = os.path.basename(chunk_path)
            parts = filename.replace("chunk_", "").replace(".mp4", "").split("_")
            start = int(parts[0])
            end = int(parts[1])
            chunk_paths.append((chunk_path, start, end))

        print(f"Reusing {len(chunk_paths)} existing chunks from {output_dir}")
        return chunk_paths

    # Otherwise, create chunks
    video = VideoFileClip(video_path)
    # duration = int(video.duration)
    duration = video.duration

    chunk_paths = []

    for start in range(0, int(duration), chunk_length): # change to int(duration) from duration
        end = min(start + chunk_length, int(duration)) # change to int(duration)

        chunk_path = os.path.join(
            output_dir,
            f"chunk_{start:04d}_{end:04d}.mp4"
        )

        if hasattr(video, "subclipped"):
            clip = video.subclipped(start, end)
        else:
            clip = video.subclip(start, end)

        clip.write_videofile(
            chunk_path,
            codec="libx264",
            audio=False,
            logger=None
        )

        clip.close()
        chunk_paths.append((chunk_path, start, end))

    video.close()
    return chunk_paths

In [7]:

# CHOOSE VIDEO
video_number = 5
video_path = f"TVSum/video_{video_number}.mp4"


# PARAMETERS
# chunk length is the lenght in seconds of each chunk
# num frames is the number of frames per chunk
# max tokens is the upper bound on number of words per generated output for each chunk

chunk_length = 15
num_frames = 10
max_new_tokens = 20

output_dir = f"video_{video_number}_chunks_{chunk_length}s"

chunks = split_video_into_chunks(
    video_path,
    output_dir=output_dir,
    chunk_length= chunk_length# changed to 8 seconds
)

all_outputs = []

for chunk_path, start, end in chunks:
    output = run_vlm_on_video(
        chunk_path,
        PROMPT_EVENT_ONLY,
        num_frames=num_frames, 
        max_new_tokens=max_new_tokens # match it to length of 1-2 sentences
    )

    all_outputs.append({
        "chunk": chunk_path,
        "start": start,
        "end": end,
        "vlm_output": output
    })

    print(f"\nCHUNK {start}-{end} seconds")
    print(output)




Reusing 8 existing chunks from video_5_chunks_15s

CHUNK 0-15 seconds
Dave Campbell is explaining the GMC Certified Service.

CHUNK 15-30 seconds
A man in a white shirt talking to the camera about a GMC truck.

CHUNK 30-45 seconds
A hand is rotating a tire on a car.

CHUNK 45-60 seconds
A man in a white shirt explains something about the GMC truck.

CHUNK 60-75 seconds
A person is working on a car with a GMC logo on it.

CHUNK 75-90 seconds
A man in a white shirt talking to the camera about a GMC truck.

CHUNK 90-105 seconds
A man in a white shirt is talking to the camera in front of a black GMC truck.

CHUNK 105-111 seconds
GMC Certified Service replacing your tires.


In [9]:
def parse_vlm_events(text):
    # Expects the output per chunk (text is for chunk output) to be normal text descritpion
    # Should work also if each chunk gives multiple events

    parsed_events = []
    parsed_events.append({
            "event": text.strip(" ,.-"),   # remove comma, dot etc.
            "start_time": None,
            "end_time": None
    })

    return parsed_events

In [10]:
# go over all chunk outputs in chronological order 
# and store a list of salient event descriptions
predicted_events = []

for item in all_outputs:
    parsed = parse_vlm_events(item["vlm_output"])
    predicted_events.extend([e["event"] for e in parsed])


In [11]:
print(predicted_events)

['Dave Campbell is explaining the GMC Certified Service', 'A man in a white shirt talking to the camera about a GMC truck', 'A hand is rotating a tire on a car', 'A man in a white shirt explains something about the GMC truck', 'A person is working on a car with a GMC logo on it', 'A man in a white shirt talking to the camera about a GMC truck', 'A man in a white shirt is talking to the camera in front of a black GMC truck', 'GMC Certified Service replacing your tires']


In [12]:
# CURRENTLY DONE MANUALLY BUT IDEALLY SHOULD HAVE A FUNCTION TO EXTRACT TEXT FROM THE ANNOTATIONS TXT


# THIS IS FOR VIDEO 5
video_annotations = [

        "Branding screen of GMC's tire replacement service",
        "Person talking in front of a GMC car",
        "Technician working on a car tire", 
        "Comparison of a tire with a good and bad thread", 
        "Displayed text recommending tire rotation every 7,500 miles",
        "Demonstration of checking tire tread depth using a tool and a coin", 
        "Tire spinning while mounted", 
        "People interacting near a car", 
        "Branding screen of GMC's tire replacement service"

]





# FOR VIDEO 6
# video_annotations = [

# 'The cameraman shows off patio with tables and chairs',
# 'The cameraman shows stuck truck',
# 'The cameraman shows the rope and the tree that are keeping the truck secure',
# 'The cameraman shows the steep ravine which the truck was about to fall in', 
# 'The second truck reversing close to the stuck truck',
# 'The cameraman shows a "drive safe" sign', 
# 'The palfinger crane picking up the stuck truck', 
# 'The truck moves on the road', 
# 'Both trucks driving off in the distance'

# ]

In [13]:
# ! pip install sentence_transformers


model_transformer = SentenceTransformer('all-MiniLM-L6-v2')

# embed ground truth and predicted event sentences
gt_embeddings = model_transformer.encode(video_annotations)
pred_embeddings = model_transformer.encode(predicted_events)


# computing pairwise cosine similarity between ground truth and predicted embeddings
sim_matrix = cosine_similarity(gt_embeddings, pred_embeddings) # maybe other way around




In [14]:
print(sim_matrix)

[[ 0.48248544  0.39244086  0.31257278  0.472232    0.5168632   0.39244086
   0.33239585  0.6828753 ]
 [ 0.35963216  0.6798316   0.28332806  0.5462053   0.5883589   0.6798316
   0.6993158   0.3035559 ]
 [ 0.22330886  0.17786506  0.5745188   0.24296385  0.4483037   0.17786506
   0.13195057  0.45478854]
 [ 0.04680708  0.08764574  0.4496262   0.11636135  0.07537089  0.08764574
   0.05828612  0.3386023 ]
 [ 0.13616705  0.15591183  0.47192127  0.1976744   0.11481188  0.15591183
   0.07965077  0.33502868]
 [ 0.11187492  0.11136028  0.43192223  0.16210973  0.15125111  0.11136028
   0.07647246  0.32758218]
 [-0.02098751  0.02680007  0.57763267  0.03888695  0.04260931  0.02680007
  -0.00866552  0.27066168]
 [ 0.09007821  0.32469422  0.30814746  0.23982835  0.3763727   0.32469422
   0.32864335  0.10660838]
 [ 0.4824854   0.39244086  0.31257278  0.47223204  0.5168631   0.39244086
   0.33239585  0.6828751 ]]


In [15]:
print(np.mean((sim_matrix)))

0.29422772


In [16]:
# THRESHOLD IS SOMETHING THAT CAN BE CHANGED (0.5 for now)

threshold = 0.5

matches = []
used_preds = set()


# THIS LOOP ENSURES THAT IF AN ANNOTATION IS MATCHED BY A PREDICTION IT CAN'T BE MATCHED BY ANOTHER ONE 
for i, gt in enumerate(video_annotations):
    
    # the predicted event that is most similar to a specific ground truth annotaiton
    best_j = np.argmax(sim_matrix[i])
    best_score = sim_matrix[i][best_j]

    # only a match if the predicted event is similar enough (higher than threshold) + it hasn't been matched to another ground truth 
    if best_score >= threshold and best_j not in used_preds: 
        matches.append((gt, predicted_events[best_j], best_score))
        used_preds.add(best_j)

In [17]:
# MAYBE NOT NEEDED

# NUMERICAL RESULTS (check if needed)

TP = len(matches)
FN = len(video_annotations) - TP
FP = len(predicted_events) - TP

precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)\




In [18]:
print(precision)
print(recall)
print(f1)

0.375
0.3333333333333333
0.35294117647058826


In [19]:
# FINDING THE MISSED EVENTS (i.e if for a ground truth event no predicted was found to be similar)

matched_gt = set([m[0] for m in matches])

missed_events = [gt for gt in video_annotations if gt not in matched_gt]

print("\nMissed events:")
for e in missed_events:
    print("-", e)


Missed events:
- Comparison of a tire with a good and bad thread
- Displayed text recommending tire rotation every 7,500 miles
- Demonstration of checking tire tread depth using a tool and a coin
- Tire spinning while mounted
- People interacting near a car


In [20]:
#  
print("250 million Model\n")

print(f"Video {video_number}")
print("Prompt given: ", PROMPT_EVENT_ONLY)

print(f"Chunk Length: {chunk_length}")
print(f"Max Tokens used (per Chunk) : {max_new_tokens}")
print(f"Number of Frames (per Chunk) :{num_frames}")

print(f"Threshold: {threshold}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")



for i in range(len(predicted_events)):
    print(f"Predicted event {i}:{predicted_events[i]}")






250 million Model

Video 5
Prompt given:   
Describe the single most important event in this video segment in one concise sentence. Focus on what is happening and who is involved.

Chunk Length: 15
Max Tokens used (per Chunk) : 20
Number of Frames (per Chunk) :10
Threshold: 0.5
Precision: 0.375
Recall:    0.333
F1-score:  0.353
Predicted event 0:Dave Campbell is explaining the GMC Certified Service
Predicted event 1:A man in a white shirt talking to the camera about a GMC truck
Predicted event 2:A hand is rotating a tire on a car
Predicted event 3:A man in a white shirt explains something about the GMC truck
Predicted event 4:A person is working on a car with a GMC logo on it
Predicted event 5:A man in a white shirt talking to the camera about a GMC truck
Predicted event 6:A man in a white shirt is talking to the camera in front of a black GMC truck
Predicted event 7:GMC Certified Service replacing your tires


## ADD STEP 4